# Extending pyvinecopulib: custom pair copulas and vines

The evaluators `pyvinecopulib.core.Bicop` / `Vinecop` and their PyTorch
counterparts `pyvinecopulib.torch.TorchBicop` / `TorchVinecop` are concrete
implementations of two array-agnostic contracts, `BicopLike` and `VinecopLike`.
You can plug your **own** pair copula into a vine by implementing the contract —
most easily by subclassing the canonical `BicopBase` / `VinecopBase`, which fill
in almost everything from a few primitives.

For almost every use case a vine is **simplified and unconditional**: each pair
copula has fixed parameters and sees only its two h-transformed arguments. That
is the default, and where this notebook starts. It then shows how to **select**
a structure from data with your own pairs, how to put them on an edge with a
**discrete** variable, how to **condition** on a subset of them, and the two
advanced extensions `VinecopBase` also
supports — a **non-simplified** vine (each pair also conditions on its edge's
conditioning set) and **external covariates**.

Everything here is plain NumPy; the same subclassing pattern works on PyTorch.

In [1]:
import numpy as np
from scipy.special import ndtr, ndtri  # standard-normal CDF / quantile

import pyvinecopulib as pv
from pyvinecopulib.core import (
  BicopBase,
  DiscretePair,
  NonSimplifiedContext,
  VinecopBase,
)

rng = np.random.default_rng(0)

## 1. A custom pair copula

Subclass `BicopBase` and implement just three primitives — `pdf`, `hfunc1`,
`hfunc2` (the density and the two conditional CDFs). `BicopBase` supplies the
rest: the inverse h-functions `hinv1` / `hinv2` (numerical), `sample`,
`loglik`, a density `plot`, and a `__repr__`.

The optional second argument `x` carries conditioning variables. We start with
the ordinary case: with `x=None` the correlation is a fixed `base_rho`, so this
is a plain Gaussian pair copula. (Section 3 switches `x` on to make it
conditional.)

In [2]:
class GaussianBicop(BicopBase[np.ndarray]):
  """Gaussian pair copula; correlation optionally depends on covariates ``x``."""

  def __init__(self, *, base_rho=0.5, scale=0.6, rho_max=0.8):
    self._base_rho, self._scale, self._rho_max = base_rho, scale, rho_max

  def _rho(self, u, x):
    if x is None:  # ordinary (simplified) pair -> a fixed correlation
      return np.full(u.shape[0], self._base_rho)
    w = np.arange(1, x.shape[1] + 1)  # position weights (see section 3)
    return self._rho_max * np.tanh(self._scale * (x * w).sum(-1) / x.shape[1])

  def pdf(self, u, *, x=None):
    uc = np.clip(u, 1e-10, 1 - 1e-10)
    z1, z2 = ndtri(uc[:, 0]), ndtri(uc[:, 1])
    r = self._rho(u, x)
    om = 1 - r * r
    return np.exp(
      (2 * r * z1 * z2 - r * r * (z1**2 + z2**2)) / (2 * om)
    ) / np.sqrt(om)

  def hfunc1(self, u, *, x=None):  # P(U2 <= u2 | U1 = u1)
    uc = np.clip(u, 1e-10, 1 - 1e-10)
    z1, z2 = ndtri(uc[:, 0]), ndtri(uc[:, 1])
    r = self._rho(u, x)
    return ndtr((z2 - r * z1) / np.sqrt(1 - r * r))

  def hfunc2(self, u, *, x=None):  # P(U1 <= u1 | U2 = u2)
    uc = np.clip(u, 1e-10, 1 - 1e-10)
    z1, z2 = ndtri(uc[:, 0]), ndtri(uc[:, 1])
    r = self._rho(u, x)
    return ndtr((z1 - r * z2) / np.sqrt(1 - r * r))

  def flip(self):  # enables hosting in structure selection (section 2.1)
    # Gaussian is exchangeable: swapping the arguments leaves it unchanged.
    return GaussianBicop(
      base_rho=self._base_rho, scale=self._scale, rho_max=self._rho_max
    )

  def _sample_uniform(self, n, qrng, seeds):  # enables sample()
    return np.random.default_rng(seeds[0] if seeds else 0).uniform(size=(n, 2))

`BicopBase` fills in the rest from those three primitives — the inverse
h-functions, a sampler, the log-likelihood, a density plot, and a repr:

In [3]:
cop = GaussianBicop(base_rho=0.6)
u = rng.uniform(0.05, 0.95, size=(5, 2))
print("pdf:     ", np.round(cop.pdf(u), 3))
print("hinv1:   ", np.round(cop.hinv1(u), 3), "(numerical inverse of hfunc1)")
print("loglik:  ", round(float(cop.loglik(u)), 3))
print("sample:  ", np.round(cop.sample(3, seeds=[1]), 3).tolist())
print(repr(cop))

pdf:      [0.952 2.689 1.68  1.281 0.91 ]
hinv1:    [0.402 0.021 0.915 0.719 0.852] (numerical inverse of hfunc1)
loglik:   1.612
sample:   [[0.512, 0.909], [0.144, 0.748], [0.312, 0.327]]
GaussianBicop()


## 2. Hosting it in a vine

`VinecopBase` implements the whole tree-by-tree cascade (`pdf`, `rosenblatt`,
`inverse_rosenblatt`, `sample`, `cdf`, `loglik`, `plot`, ...) on top of a
single required hook, `_get_pair_copula`, which returns the pair at
`(tree, edge)`. By default it assembles nothing extra per edge
(`SimplifiedContext`), i.e. an ordinary unconditional vine.

In [4]:
class ListVinecop(VinecopBase[np.ndarray]):
  """A vine over a plain nested list of BicopLike pairs."""

  def __init__(self, pairs, structure, context=None):
    self._pairs = pairs
    self._bind_vine(structure, context)  # SimplifiedContext by default

  def _get_pair_copula(self, tree, edge):
    return self._pairs[tree][edge]

  def _sample_uniform(self, n, qrng, seeds):
    return np.random.default_rng(seeds[0] if seeds else 0).uniform(
      size=(n, self.d)
    )


d = 4
structure = pv.RVineStructure.from_order([1, 2, 3, 4])
pairs = [
  [GaussianBicop(base_rho=r) for r in row]
  for row in ([0.5, 0.4, 0.3], [0.25, 0.2], [0.15])
]
vine = ListVinecop(pairs, structure)  # simplified + unconditional
print(vine, "| dim:", vine.dim, "| trees:", vine.trunc_lvl)

U = rng.uniform(0.05, 0.95, size=(1000, d))
print("log-likelihood:", round(float(vine.loglik(U)), 2))
sim = vine.sample(500, seeds=[0])
print("sample ->", sim.shape)

ListVinecop(dim=4, trunc_lvl=3, order=[1, 2, 3, 4]) | dim: 4 | trees: 3
log-likelihood: -99.97
sample -> (500, 4)


A simplified vine of Gaussian pairs is exactly what the built-in `pv.Vinecop`
builds, so hosting `pv.Bicop` Gaussian pairs in our `VinecopBase` subclass
reproduces `pv.Vinecop.from_structure` to machine precision:

In [5]:
g = pv.families.gaussian
gauss = [
  [pv.Bicop(family=g, parameters=np.array([[r]])) for r in row]
  for row in ([0.5, 0.4, 0.3], [0.25, 0.2], [0.15])
]
ours = ListVinecop(gauss, structure)
ref = pv.Vinecop.from_structure(structure=structure, pair_copulas=gauss)
print("max |pdf difference|:", float(np.abs(ours.pdf(U) - ref.pdf(U)).max()))

max |pdf difference|: 0.0


### 2.1 Selecting a structure from data

So far the structure was given. `VinecopBase.select` builds one from data,
running the same array-agnostic Dissmann selection `pv.Vinecop` uses: weight
each candidate edge by Kendall's τ, keep a maximum-dependence spanning tree, fit
its pairs, and propagate their h-functions to the next tree. It returns the
chosen `RVineStructure` **together with** the fitted pairs, reused — and
reoriented via each pair's `flip` — onto their finalized slots, so nothing is
re-fit. You supply `fit_edge(tree, edge, u_e, x_e)`, which fits one pair per
edge; here it estimates a Gaussian correlation. (A pair hosted in selection
therefore also needs a `flip`; evaluation along a fixed structure never calls
it.)

In [6]:
def fit_edge(tree, edge, u_e, x_e):
  z = ndtri(np.clip(u_e, 1e-10, 1 - 1e-10))  # normal scores
  rho = float(np.corrcoef(z, rowvar=False)[0, 1])
  return GaussianBicop(base_rho=rho)


data = pv.to_pseudo_obs(
  rng.standard_normal((2000, d)) @ rng.standard_normal((d, d))
)
sel_structure, sel_pairs = VinecopBase.select(
  data, fit_edge, tree_criterion="tau"
)
selected = ListVinecop(sel_pairs, sel_structure)
print(
  "selected order:",
  list(sel_structure.order),
  "| trees:",
  sel_structure.trunc_lvl,
)
print("log-likelihood:", round(float(selected.loglik(data)), 1))

selected order: [4, 2, 3, 1] | trees: 3
log-likelihood: 1000.8


### 2.2 Discrete variables

A discrete variable is not described by one number but by two: the value of its
distribution function, $F(x)$, and its left limit, $F(x^-)$. The mass of the
atom the observation fell in is the gap between them, and every copula quantity
that is a derivative in a continuous argument becomes a difference quotient over
that gap.

You declare which variables have atoms when you bind the vine, and pass the
extra columns with the data — either the expanded $n \times 2d$ layout (all
$d$ values, then all $d$ left limits) or the compact $n \times (d + k)$ one,
which appends only the $k$ discrete variables' left limits. That is the same
input `pv.Vinecop` takes.

The pair copulas themselves stay continuous. The vine works out which of them
sees a discrete argument — `pair_var_types(tree, edge)`, derived from the
structure — and `DiscretePair` turns a continuous pair into the mixed-discrete
one that edge needs, out of its `pdf`, `cdf`, `hfunc1` and `hfunc2`. So the only
thing a custom pair copula has to add is a `cdf`.

In [ ]:
class GumbelBicop(BicopBase[np.ndarray]):
  """A Gumbel pair copula: closed form all the way down to its `cdf`."""

  def __init__(self, theta):
    self.theta = float(theta)

  def _parts(self, u):
    # Trim as the built-in families do: an h-function feeding the next tree can
    # land exactly on 0 or 1, where -log is undefined.
    uc = np.clip(u, 1e-10, 1 - 1e-10)
    l1, l2 = -np.log(uc[:, 0]), -np.log(uc[:, 1])
    a = (l1**self.theta + l2**self.theta) ** (1 / self.theta)
    return uc[:, 0], uc[:, 1], l1, l2, a

  def cdf(self, u, *, x=None):  # what a discrete edge needs
    return np.exp(-self._parts(u)[4])

  def pdf(self, u, *, x=None):
    t = self.theta
    u1, u2, l1, l2, a = self._parts(u)
    return (
      np.exp(-a)
      / (u1 * u2)
      * a ** (1 - 2 * t)
      * (l1 * l2) ** (t - 1)
      * (a + t - 1)
    )

  def hfunc1(self, u, *, x=None):
    t = self.theta
    u1, _, l1, _, a = self._parts(u)
    return np.exp(-a) * a ** (1 - t) * l1 ** (t - 1) / u1

  def hfunc2(self, u, *, x=None):
    t = self.theta
    _, u2, _, l2, a = self._parts(u)
    return np.exp(-a) * a ** (1 - t) * l2 ** (t - 1) / u2

  def flip(self):  # Gumbel is exchangeable
    return GumbelBicop(self.theta)


class DiscreteListVinecop(ListVinecop):
  """`ListVinecop` that also knows which of its variables have atoms."""

  def __init__(self, pairs, structure, var_types):
    self._pairs = pairs
    self._bind_vine(structure, var_types=var_types)

  def _get_pair_copula(self, tree, edge):
    types = self.pair_var_types(tree, edge)
    pair = self._pairs[tree][edge]
    return pair if "d" not in types else DiscretePair(pair, types)


# Variable 1 is a Binomial(4, 0.5) count, described by F(x) and F(x-).
cdf = np.cumsum([1, 4, 6, 4, 1]) / 16.0
counts = rng.binomial(4, 0.5, 800)
u_cont = rng.uniform(0.02, 0.98, size=(800, 2))
disc_data = np.column_stack(
  [cdf[counts], u_cont, np.where(counts > 0, cdf[counts - 1], 0.0)]
)

var_types = ["d", "c", "c"]
thetas = ([2.0, 1.5], [1.3])
disc_structure = pv.RVineStructure.from_order([1, 2, 3])
disc_vine = DiscreteListVinecop(
  [[GumbelBicop(t) for t in row] for row in thetas],
  disc_structure,
  var_types,
)
reference = pv.Vinecop.from_structure(
  structure=disc_structure,
  pair_copulas=[
    [
      pv.Bicop.from_family(pv.families.gumbel, parameters=np.array([[t]]))
      for t in row
    ]
    for row in thetas
  ],
  var_types=var_types,
)
print("compact layout:", disc_data.shape, "= (n, d + k) with d = 3, k = 1")
print("tree 0, edge 0 sees:", disc_vine.pair_var_types(0, 0))
print("tree 1, edge 0 sees:", disc_vine.pair_var_types(1, 0))
print(
  "max |pdf difference| vs pv.Vinecop:",
  f"{np.abs(disc_vine.pdf(disc_data) - reference.pdf(disc_data)).max():.1e}",
)

Selection works the same way: pass `var_types` to `VinecopBase.select`, and
each edge that has a discrete argument gets a four-column `u_e` plus the keyword
`var_types` telling the callback which of its two arguments has atoms. Edges
without one are called exactly as before, so a callback written for a continuous
vine keeps working.

In [ ]:
def disc_fit_edge(tree, edge, u_e, x_e, var_types=("c", "c")):
  tau = float(pv.utils.wdm(u_e[:, 0], u_e[:, 1], "tau"))
  return GumbelBicop(1 / (1 - min(abs(tau), 0.95)))


disc_sel_structure, disc_sel_pairs = VinecopBase.select(
  disc_data, disc_fit_edge, var_types=var_types
)
disc_selected = DiscreteListVinecop(
  disc_sel_pairs, disc_sel_structure, var_types
)
print("selected order:", list(disc_sel_structure.order))
print("log-likelihood:", round(float(disc_selected.loglik(disc_data)), 2))

### 2.3 Conditioning on a subset of the variables

Sampling from the *conditional* distribution of some variables given fixed
values of the others needs the conditioning set to sit at the **tail** of the
vine order: in natural order each Rosenblatt coordinate reads only the columns
at or after it, so the tail is a self-contained sub-vine and can be inverted on
its own.

Two things arrange that, and both work on your own pairs. `VinecopBase.select`
takes a `conditioning_set` and selects a vine whose order ends with it, and
`reorient` relabels an already-fitted vine to an equivalent one with a chosen
tail — same density, same log-likelihood, different sampling order. Unlike
`pv.Vinecop.reorient`, which mutates, it *returns* the relabeled structure and
pairs, because `VinecopBase` does not own pair storage. Both move pair copulas
between slots, so they need `flip` (section 1).

In [ ]:
cond_structure, cond_pairs = VinecopBase.select(
  data, fit_edge, conditioning_set=[1, 2]
)
cond_vine = ListVinecop(cond_pairs, cond_structure)
print("selected order:", list(cond_structure.order), "-> tail is {1, 2}")

# One conditioning point per output row: variables 1 and 2 held at (0.3, 0.8).
u_cond = np.repeat([[0.3, 0.8]], 500, axis=0)
drawn = cond_vine.sample_conditional(u_cond, seeds=[0])
print("sample_conditional ->", drawn.shape)
print("conditioning columns reproduced:", np.allclose(drawn[:, :2], u_cond))
print(
  "free variables shift with the condition:",
  np.round(drawn[:, 2:].mean(axis=0), 3),
  "vs unconditional",
  np.round(cond_vine.sample(500, seeds=[0])[:, 2:].mean(axis=0), 3),
)

# Not every set is an admissible tail of a vine that was not selected for it --
# that is what `conditioning_set` above buys. On the section-2.1 vine:
try:
  selected.reorient([1, 2])
except RuntimeError as err:
  print("\nreorient([1, 2]) on the plain vine:", str(err).split(";")[0])

# Single variables usually are admissible, and relabeling preserves the model.
movable = []
for v in list(selected.order)[:-1]:  # skip the one already last
  try:
    selected.reorient([int(v)])
  except RuntimeError:
    continue
  movable.append(int(v))
print("variables this vine can move to the tail:", movable)

structure2, pairs2 = selected.reorient(movable[:1] or [int(selected.order[-1])])
relabeled = ListVinecop(pairs2, structure2)
print("new order:", list(structure2.order))
print(
  "log-likelihood unchanged:",
  round(float(selected.loglik(data)), 6),
  "->",
  round(float(relabeled.loglik(data)), 6),
)

## 3. Advanced: non-simplified and conditional vines

The two extensions below cover the ~1% of cases that need them; a plain vine
never has to touch them.

**Non-simplified.** Pass a `NonSimplifiedContext`: each pair copula `c_{a,b;D}`
then also receives its edge's conditioning-set values `u_D` (assembled by the
cascade) as its `x`, so its correlation varies with the conditioning set. Our
`GaussianBicop` already reads `x` in its `_rho` link, so the same kind of pair
becomes a genuinely non-simplified vine just by switching the context. Because
the conditioning variables are finalized before they are needed,
`inverse_rosenblatt` still inverts `rosenblatt` exactly:

In [7]:
cond_pairs = [
  [GaussianBicop(scale=0.6) for _ in range(d - 1 - t)] for t in range(d - 1)
]
cond_vine = ListVinecop(cond_pairs, structure, NonSimplifiedContext())

W = cond_vine.rosenblatt(U)  # dependent -> independent uniforms
U_back = cond_vine.inverse_rosenblatt(W)  # and back
print("non-simplified round-trip max error:", float(np.abs(U_back - U).max()))
print("log-likelihood:", round(float(cond_vine.loglik(U)), 2))

non-simplified round-trip max error: 6.661338147750939e-15
log-likelihood: -223.9


**External covariates.** You can also pass an external covariate matrix `x`
(row-aligned with `u`) to any evaluator; every pair sees it appended to its
conditioning matrix. The density then shifts with the covariates:

In [8]:
X = rng.standard_normal(size=(1000, 2))
mean_at_zero = cond_vine.pdf(U, x=np.zeros((1000, 2))).mean()
mean_at_x = cond_vine.pdf(U, x=X).mean()
print("mean pdf, x = 0 :", round(float(mean_at_zero), 3))
print("mean pdf, x ~ N :", round(float(mean_at_x), 3))

mean pdf, x = 0 : 1.01
mean pdf, x ~ N : 1.087


## What's next

- The same subclassing pattern works on PyTorch: implement the primitives with
  `torch` ops (and make the class an `nn.Module`) for autograd and GPU.
- `VinecopBase.select` (section 2.1) chooses a *simplified* structure and fits
  its pairs in one pass. To fit pairs along a **fixed** structure — including a
  **non-simplified** one, edge by edge — drive
  `VinecopBase.fit(structure, u, fit_edge, context=..., x=...)`.
- A pair copula needs two optional pieces beyond the three primitives, each for
  one job: `flip` to be reused in **selection** (section 2.1) or moved by a
  **relabeling** (section 2.3), and `cdf` to sit on a **discrete** edge (section
  2.2). `BicopBase` raises for both until you provide them.
- See `pyvinecopulib.core.BicopBase` / `VinecopBase`, `BicopLike` /
  `VinecopLike`, `DiscretePair`, and `SimplifiedContext` /
  `NonSimplifiedContext` for the full contracts.